# 06 — Evaluating Text Generation Quality: BLEU and Perplexity

## 📚 Learning Objectives

By completing this notebook, you will:
- **Compute BLEU scores** with NLTK and see them track candidate quality
- **Compute perplexity** of a language model you train — before training, after training, and on out-of-domain text
- Interpret both metrics and know their blind spots

## 🔗 Prerequisites

- ✅ Example 01 (the char-level LM — its perplexity is measured here)
- ✅ Unit 1 example 10 (metric concepts: the FID/BLEU overview)

---

## Introduction

Two numbers dominate text-generation evaluation:

- **BLEU** (0–1, higher better): n-gram overlap between a candidate text and reference text(s), with a brevity penalty. Built for machine translation; still a standard baseline metric.
- **Perplexity** (≥1, lower better): `exp(average cross-entropy)` of a language model on a text. Intuition: "on average, the model was as uncertain as if choosing uniformly among *PPL* options at each step." A model that knows nothing about a 27-character alphabet has perplexity ≈ 27; training pushes it down.


In [1]:
# WHAT/WHY: compute real BLEU scores on candidates of graded quality and
# check that the scores fall as quality falls.
%pip install nltk -q
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smooth = SmoothingFunction().method1     # standard smoothing for short sentences

# ── Reference "translation" and candidates from best to worst ─────────────
reference = [["machine", "translation", "quality", "improves", "every", "single", "year"]]
candidates = {
    "exact match":      ["machine", "translation", "quality", "improves", "every", "single", "year"],
    "near paraphrase":  ["machine", "translation", "quality", "improves", "each", "single", "year"],
    "partial overlap":  ["translation", "quality", "is", "getting", "better"],
    "unrelated":        ["the", "cat", "enjoys", "sleeping", "on", "warm", "windowsills"],
}

# ── Score each candidate with 4-gram BLEU ─────────────────────────────────
print("candidate          BLEU (0-1, higher is better)")
for name, cand in candidates.items():
    score = sentence_bleu(reference, cand, smoothing_function=smooth)
    print(f"{name:<18} {score:.4f}")

print("\nThe scores fall with quality — and note the blind spot: a *correct*")
print("rewording with different words would also score low. BLEU measures")
print("overlap with the reference, not meaning.")


Note: you may need to restart the kernel to use updated packages.


candidate          BLEU (0-1, higher is better)
exact match        1.0000
near paraphrase    0.4889
partial overlap    0.0762
unrelated          0.0000

The scores fall with quality — and note the blind spot: a *correct*
rewording with different words would also score low. BLEU measures
overlap with the reference, not meaning.


## Perplexity — Measured on a Model We Train

We train the example-01 char-LM on a deliberately **repetitive** corpus (three sentences repeated ten times), holding out the final 20% as **validation** — because the corpus repeats, the validation slice contains the same sentence patterns the model trained on. Perplexity = `exp(average next-char cross-entropy)`. Three measurements tell the story:

1. **Untrained model** on validation text — should be ≈ vocabulary size (pure guessing)
2. **Trained model** on validation text — close to 1 here, because the repetitive corpus is essentially memorizable. (On real, varied corpora a good char model lands in the tens — perplexity 1 means "always certain and right", achievable only when the text repeats.)
3. **Trained model** on out-of-domain text — enormous: the model is now *confidently wrong* about text unlike its training data. Perplexity measures fit *between a model and a text*, which is exactly why LM papers compare models on one fixed test set


In [2]:
# WHAT/WHY: compute perplexity = exp(avg cross-entropy) of a char-LM at three
# stages — untrained, trained (in-domain), and trained (out-of-domain) — to
# see what the number responds to.
import torch, torch.nn as nn, torch.optim as optim
import numpy as np

torch.manual_seed(42)
# ── Repetitive training corpus: three sentences × 10 ──────────────────────
sentences = ("the cat sat on the mat and watched the birds outside. "
             "the dog ran across the park chasing a red ball. "
             "the birds sang in the tall green tree every morning. ")
text = sentences * 10
ood_text = "quarterly revenue exceeded forecasts with cloud services demand up strongly"

# ── Vocabulary over both texts, then an 80/20 train/validation split ──────
chars = sorted(set(text + ood_text))
c2i = {c: i for i, c in enumerate(chars)}
VOCAB = len(chars); SEQ_LEN = 20
split = int(len(text) * 0.8)
train_text, val_text = text[:split], text[split:]
print(f"vocabulary: {VOCAB} chars — untrained perplexity should be ≈ {VOCAB}")

def make_dataset(t):
    enc = [c2i[c] for c in t]
    X = [enc[i:i+SEQ_LEN] for i in range(len(enc) - SEQ_LEN - 1)]
    y = [enc[i+SEQ_LEN]   for i in range(len(enc) - SEQ_LEN - 1)]
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

X_tr, y_tr   = make_dataset(train_text)
X_val, y_val = make_dataset(val_text)
X_ood, y_ood = make_dataset(ood_text)

class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        self.lstm  = nn.LSTM(32, 128, batch_first=True, num_layers=2)
        self.fc    = nn.Linear(128, VOCAB)
    def forward(self, x):
        out, _ = self.lstm(self.embed(x))
        return self.fc(out[:, -1, :])

model = CharLM(); loss_fn = nn.CrossEntropyLoss()

def perplexity(X, y):
    """exp of the average next-char cross-entropy — the definition, verbatim."""
    model.eval()
    with torch.no_grad():
        return float(torch.exp(loss_fn(model(X), y)))

# ── Measurement 1: untrained model (should be ≈ VOCAB, i.e., guessing) ────
ppl_untrained = perplexity(X_val, y_val)
print(f"\n1. untrained model, validation text:  PPL = {ppl_untrained:7.1f}")

# ── Train on the first 80% of the corpus ──────────────────────────────────
opt = optim.Adam(model.parameters(), lr=3e-3)
for step in range(400):
    model.train()
    perm = torch.randperm(len(X_tr))[:256]
    loss = loss_fn(model(X_tr[perm]), y_tr[perm])
    opt.zero_grad(); loss.backward(); opt.step()

# ── Measurements 2 and 3: trained model, in-domain vs out-of-domain ───────
ppl_val = perplexity(X_val, y_val)
ppl_ood = perplexity(X_ood, y_ood)
print(f"2. trained model, validation text:    PPL = {ppl_val:7.2f}")
print(f"3. trained model, out-of-domain text: PPL = {ppl_ood:7.1f}")

print("\nRead the three numbers together: untrained ≈ vocabulary size (guessing);")
print("training collapsed the uncertainty on in-domain text (≈ 1 only because the")
print("corpus repeats — real corpora land in the tens); and out-of-domain text is")
print("WORSE than guessing — the model is confidently wrong there. Perplexity")
print("measures fit BETWEEN a model and a text, not text quality by itself.")


vocabulary: 26 chars — untrained perplexity should be ≈ 26

1. untrained model, validation text:  PPL =    26.2


2. trained model, validation text:    PPL =    1.00
3. trained model, out-of-domain text: PPL =   671.0

Read the three numbers together: untrained ≈ vocabulary size (guessing);
training collapsed the uncertainty on in-domain text (≈ 1 only because the
corpus repeats — real corpora land in the tens); and out-of-domain text is
WORSE than guessing — the model is confidently wrong there. Perplexity
measures fit BETWEEN a model and a text, not text quality by itself.


## 📚 References & Further Reading

**Papers:**
- Papineni et al. (2002) — [BLEU](https://aclanthology.org/P02-1040/)
- Jelinek et al. (1977) — perplexity's origin in speech recognition; see also [The Gradient — Understanding Perplexity](https://thegradient.pub/understanding-evaluation-metrics-for-language-models/)

**Beyond BLEU/perplexity:** ROUGE (summarization), BERTScore (semantic similarity), and LLM-as-judge evaluations are today's standard complements.


## 📝 Summary

In **06 — Evaluating Text Generation Quality** you computed both metrics for real: BLEU scores that fell in step with candidate quality (printed table), and perplexity measured three ways on a model you trained — ≈ vocabulary size when untrained, far lower on in-domain validation text after training, and high again on out-of-domain text. Together with Unit 1's FID work (example 10), you now have the standard evaluation toolkit for generative models.
